In [ ]:
"""
Product Demand Forecasting Pipeline
===================================
Author: [본인 이름/닉네임]
Description:
    CatBoost Regressor와 SARIMAX 모델을 결합한 하이브리드 시계열 수요 예측 파이프라인입니다.
    품목별 판매량(Volume)에 따라 모델을 이원화(High/Low Bucket)하여 예측 정확도를 최적화합니다.
"""

from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ==========================================
# CONSTANTS & PATH CONFIGURATION
# ==========================================
FP_ORDER = Path("order_train.csv")
FP_DETAIL = Path("order_detail_train.csv")
FP_TEMPLATE = Path("submission_template.xlsx")
FP_OUT = Path("submission39.csv")

GUIDE_SKU = "첫 발주 안내서"
SEASON_ANCHOR_GAMMA = 0.25
SEASON_CLIP = (0.85, 1.15)

# 12월 연말 수요 변동성 대응을 위한 그룹별 가중치 계수
DEC_GROUP_WEIGHT = {
    "liquor": 1.60,       # 연말 수요 탄력성 높음
    "mixer": 1.12,        # 음료 및 가니시류
    "promo": 1.10,        # 판촉물 및 프로모션 상품
    "tools_glass": 0.90,  # 자재 및 기물류 (완만한 수요)
    "tools": 0.98,
    "other": 1.00
}


# ==========================================
# UTILS & PREPROCESSING FUNCTIONS
# ==========================================
def norm_text(text: any) -> str:
    """텍스트 데이터를 문자열로 변환하고 공백을 제거합니다."""
    return str(text).strip() if pd.notna(text) else ""


def month_start(timestamp: any) -> pd.Timestamp:
    """날짜 데이터를 해당 월의 시작일(1일) 날짜형식으로 변환합니다."""
    return pd.to_datetime(str(timestamp)[:7] + "-01", errors="coerce")


def clamp(value: float, lo: float, hi: float) -> float:
    """입력 값을 지정된 최소(lo) 및 최대(hi) 범위 내로 제한합니다."""
    return float(min(max(float(value), lo), hi))


def monthly_series_from_hist(hist_df: pd.DataFrame, value_col: str, exog_cols: list = None) -> tuple:
    """
    이력 데이터에서 월별 시계열 데이터 및 외생변수(Exogenous Features)를 추출합니다.

    Args:
        hist_df (pd.DataFrame): 대상 이력 데이터프레임
        value_col (str): 시계열 대상 컬럼명 (Target)
        exog_cols (list, optional): 외생변수로 사용할 컬럼 리스트

    Returns:
        tuple: (pd.Series [월별 시계열], pd.DataFrame or None [외생변수 데이터프레임])
    """
    if exog_cols is None:
        exog_cols = []
        
    hist_u = (hist_df.sort_values("target_month")
                     .groupby("target_month", as_index=False)
                     .last())
    
    series = hist_u.set_index("target_month")[value_col].asfreq("MS")
    
    exog_df = None
    if exog_cols:
        cols = [c for c in exog_cols if c in hist_u.columns]
        if cols:
            exog_df = hist_u.set_index("target_month")[cols].asfreq("MS").fillna(0)
            
    return series, exog_df


def classify_product(name: str) -> tuple:
    """
    상품명을 기반으로 대분류(Group) 및 세분류(Category)를 정의합니다.

    Args:
        name (str): 상품명 컬럼 값

    Returns:
        tuple: (대분류 [group], 세분류 [category])
    """
    n = norm_text(name)

    # 분류 규칙 키워드 정의
    liquor_keys = ["1L", "유자우롱츄하이", "블루하와이", "플라워브리즈", "이츠리치", "미도리사워"]
    recipe_keys = ["레시피 패키지"]
    water_keys = ["탄산수"]
    garnish_keys = ["레몬 슬라이스", "건조 레몬", "건조 오렌지", "레드커런트", "체리", "식용꽃", "슬라이스", "칵테일 체리", "냉동"]
    herb_keys = ["로즈마리", "애플민트", "타임", "허브"]
    pump_keys = ["펌프"]
    jigger_keys = ["지거"]
    spoon_keys = ["스푼"]
    muddle_keys = ["머들러"]
    mat_keys = ["바 매트"]
    pick_keys = ["칵테일픽"]
    glass_keys = ["글라스", "하이볼", "머그", "라운드", "스트레이트", "슬림", "트라이탄"]
    menu_keys = ["메뉴판", "리피즈 메뉴판"]
    poster_keys = ["포스터"]
    banner_keys = ["배너", "X 배너", "시안"]
    guide_keys = ["첫 발주 안내서"]

    # 계층적 우선순위 기반 라우팅
    if any(k in n for k in guide_keys):
        return "promo", "guide_doc"
    if any(k in n for k in liquor_keys):
        return "liquor", "liqueur_base"
    if any(k in n for k in recipe_keys):
        return "liquor", "recipe_package"
    if any(k in n for k in water_keys):
        return "mixer", "water"
    if any(k in n for k in herb_keys):
        return "mixer", "herb"
    if any(k in n for k in garnish_keys):
        return "mixer", "garnish"
    if any(k in n for k in glass_keys):
        return "tools_glass", "glass_highball"
    if any(k in n for k in pump_keys):
        return "tools", "pump"
    if any(k in n for k in jigger_keys):
        return "tools", "jigger"
    if any(k in n for k in spoon_keys):
        return "tools", "spoon"
    if any(k in n for k in muddle_keys):
        return "tools", "muddler"
    if any(k in n for k in mat_keys):
        return "tools", "bar_mat"
    if any(k in n for k in pick_keys):
        return "tools", "cocktail_pick"
    if any(k in n for k in (menu_keys + poster_keys + banner_keys)):
        return "promo", "print"

    return "other", "other"


# ==========================================
# DATA LOADING & FEATURE ENGINEERING
# ==========================================
def load_and_prepare() -> pd.DataFrame:
    """주문 마스터 및 상세 데이터를 로드하고 병합 및 전처리를 수행합니다."""
    # 주문 원장 로드 및 컬럼 표준화
    order = pd.read_csv(FP_ORDER).rename(columns={
        "주문자 ID": "member_id", "주문일시": "order_dt", "주문번호": "order_no",
        "실 결제금액": "paid_amt", "쿠폰사용금액": "coupon_amt", "주문금액": "order_amt"
    })
    order["order_dt"] = pd.to_datetime(order["order_dt"], errors="coerce")
    for c in ["paid_amt", "coupon_amt", "order_amt"]:
        order[c] = pd.to_numeric(order[c], errors="coerce").fillna(0)

    # 주문 상세 내역 로드 및 컬럼 표준화
    det = pd.read_csv(FP_DETAIL).rename(columns={
        "주문번호": "order_no", "결제 날짜시간": "pay_dt", "상품명": "product_id",
        "판매 수량": "qty", "상품 구매 금액": "prod_amt", "상품 가격": "unit_price"
    })
    det["pay_dt"] = pd.to_datetime(det["pay_dt"], errors="coerce")
    det["product_id"] = det["product_id"].astype(str).map(norm_text)
    for c in ["qty", "prod_amt", "unit_price"]:
        det[c] = pd.to_numeric(det[c], errors="coerce").fillna(0)

    # 데이터 정합성 검증 및 병합
    df = det.merge(order[["order_no", "order_dt", "coupon_amt", "member_id"]], on="order_no", how="left")
    df["dt"] = df["pay_dt"].fillna(df["order_dt"])
    df = df.dropna(subset=["dt"])
    df = df[df["qty"] > 0]
    df["date"] = pd.to_datetime(df["dt"].dt.date)

    # 파생 변수 생성 (매출액 보정 및 쿠폰 할인 금액 배분)
    df["line_amt"] = df["prod_amt"].where(df["prod_amt"].notna(), df["unit_price"] * df["qty"])
    df["coupon_amt"] = pd.to_numeric(df["coupon_amt"], errors="coerce").fillna(0)
    
    order_sum = df.groupby("order_no")["line_amt"].transform(lambda s: s.sum() if s.sum() > 0 else 1.0)
    df["coupon_line"] = df["coupon_amt"] * (df["line_amt"] / order_sum)

    # 상품 규칙 기반 카테고리 매핑
    groups, cats = [], []
    for name in df["product_id"].astype(str):
        g, c = classify_product(name)
        groups.append(g)
        cats.append(c)
    df["group"] = groups
    df["category"] = cats
    
    return df


def make_monthly(df: pd.DataFrame) -> pd.DataFrame:
    """일별 거래 내역을 월별 집계 데이터로 집계 및 다운샘플링합니다."""
    daily = df.groupby(["product_id", "group", "category", "date"], as_index=False).agg(
        qty=("qty", "sum"),
        coupon=("coupon_line", "sum"),
        rev=("line_amt", "sum")
    )
    daily["target_month"] = pd.to_datetime(daily["date"].values.astype("datetime64[M]"))
    
    mon = (daily.groupby(["product_id", "group", "category", "target_month"], as_index=False)
                .agg(qty_month=("qty", "sum"),
                     coupon_month=("coupon", "sum"),
                     rev_month=("rev", "sum")))
                     
    mon["avg_price"] = np.where(mon["qty_month"] > 0, mon["rev_month"] / mon["qty_month"], np.nan)
    mon["promo_intensity"] = np.where(mon["rev_month"] > 0, mon["coupon_month"] / mon["rev_month"], 0.0)
    mon["promo_intensity"] = mon["promo_intensity"].fillna(0.0)
    
    return mon


def add_features(mon: pd.DataFrame) -> pd.DataFrame:
    """예측 모델에 활용할 시차 변수(Lag) 및 이동 평균(Rolling) 피처를 생성합니다."""
    mon = mon.sort_values(["product_id", "target_month"]).copy()

    # 상위 그룹 및 카테고리의 평균 판매량 메타 피처 생성
    cat_meta = mon.groupby("category")["qty_month"].mean().rename("cat_mean_qty").reset_index()
    grp_meta = mon.groupby("group")["qty_month"].mean().rename("grp_mean_qty").reset_index()
    mon = mon.merge(cat_meta, on="category", how="left")
    mon = mon.merge(grp_meta, on="group", how="left")

    def _generate_sku_features(g):
        g = g.sort_values("target_month").copy()
        # 시차(Lag) 피처 생성
        for L in [1, 2, 3]:
            g[f"lag_{L}"] = g["qty_month"].shift(L)
        # 이동 평균(Rolling Mean) 피처 생성
        for W in [3, 6]:
            g[f"roll_mean_{W}"] = g["qty_month"].rolling(W).mean()
            
        g["coupon_lag_1"] = g["coupon_month"].shift(1)
        g["promo_lag_1"] = g["promo_intensity"].shift(1)
        g["price_lag_1"] = g["avg_price"].shift(1)
        
        # 시간 관련 피처 추출
        g["year"] = g["target_month"].dt.year.astype(int)
        g["month"] = g["target_month"].dt.month.astype(int)
        g["is_year_end"] = (g["month"] == 12).astype(int)
        return g

    mon = mon.groupby("product_id", group_keys=False).apply(_generate_sku_features)
    return mon


def mark_volume(mon: pd.DataFrame, thr: float = 3.0, min_m: int = 6) -> pd.DataFrame:
    """품목별 평균 판매량 및 이력을 기준으로 High/Low 수요 버킷을 분류합니다."""
    stat = mon.groupby("product_id")["qty_month"].agg(["mean", "count"]).reset_index()
    stat["bucket"] = np.where((stat["mean"] >= thr) & (stat["count"] >= min_m), "high", "low")
    return stat[["product_id", "bucket"]]


# ==========================================
# MACHINE LEARNING & STATISTICAL MODELS
# ==========================================
def train_catboost(train_df: pd.DataFrame, cat_feature_names: list) -> CatBoostRegressor:
    """High-volume 품목을 타겟으로 머신러닝 기반 CatBoost 모델을 학습시킵니다."""
    X = train_df.drop(columns=["qty_month", "product_id", "target_month"])
    y = train_df["qty_month"].values
    X = X.fillna(0)

    # 범주형 피처 인덱스 정의
    cat_features = [c for c in cat_feature_names if c in X.columns]
    cat_idx = [X.columns.get_loc(c) for c in cat_features]

    # 특정 유효 월(예: 2023-11 부분월)에 대한 가중치 다운웨이트 적용
    w = np.ones(len(train_df), dtype=float)
    w = np.where(train_df["target_month"] == pd.Timestamp("2023-11-01"), 0.5, w)

    model = CatBoostRegressor(
        iterations=1100, depth=8, learning_rate=0.05,
        loss_function="RMSE", random_seed=42, verbose=0
    )
    model.fit(X, y, cat_features=cat_idx, sample_weight=w)
    model._feature_cols = X.columns.tolist()
    return model


def predict_catboost(model: CatBoostRegressor, row_dict: dict) -> float:
    """학습된 CatBoost 모델을 통해 단일 시점 추론을 수행합니다."""
    X = pd.DataFrame([row_dict]).fillna(0)
    for c in model._feature_cols:
        if c not in X.columns:
            X[c] = 0
    X = X[model._feature_cols].fillna(0)
    return float(model.predict(X)[0])


def forecast_sarimax(y_ser: pd.Series, exog_df: pd.DataFrame, horizon_months: list) -> dict:
    """통계적 시계열 알고리즈인 SARIMAX를 통해 장기/Low-volume 수요를 예측합니다."""
    result = {}
    try:
        mod = SARIMAX(y_ser, exog=exog_df,
                      order=(0, 1, 1), seasonal_order=(1, 1, 0, 12),
                      enforce_stationarity=False, enforce_invertibility=False)
        fit = mod.fit(disp=False)
        
        last_ex = exog_df.iloc[-1:].copy() if exog_df is not None else None
        fut_ex = pd.concat([last_ex] * len(horizon_months), ignore_index=True) if last_ex is not None else None
        
        fc = fit.forecast(steps=len(horizon_months), exog=fut_ex)
        for i, tm in enumerate(horizon_months):
            result[tm] = float(fc.iloc[i])
    except Exception:
        # 모델 수렴 실패 시 최근 3개월 실적 평균으로 Fallback 처리
        nz = y_ser[y_ser > 0].tail(3)
        base = float(nz.mean() if len(nz) > 0 else 0.0)
        for tm in horizon_months:
            result[tm] = base
            
    return result


# ==========================================
# POST-PROCESSING & CALIBRATION
# ==========================================
def build_uplift_maps(mon: pd.DataFrame) -> tuple:
    """과거 동월 대비 증감률을 계산하여 연말/연초 Uplift 인덱스 맵을 구축합니다."""
    sku_ratio, sku_novq = {}, {}
    for pid, g in mon.groupby("product_id"):
        gm = g.groupby("target_month", as_index=True)["qty_month"].sum()
        nov = float(gm.get(pd.Timestamp("2023-11-01"), np.nan))
        dec = float(gm.get(pd.Timestamp("2023-12-01"), np.nan))
        if np.isfinite(nov) and nov > 0 and np.isfinite(dec):
            sku_ratio[pid] = clamp(dec / nov, 0.85, 1.30)
            sku_novq[pid] = nov
            
    cat_ratio = {}
    for cat, g in mon.groupby("category"):
        gm = g.groupby("target_month", as_index=True)["qty_month"].sum()
        nov = float(gm.get(pd.Timestamp("2023-11-01"), np.nan))
        dec = float(gm.get(pd.Timestamp("2023-12-01"), np.nan))
        if np.isfinite(nov) and nov > 0 and np.isfinite(dec):
            cat_ratio[cat] = clamp(dec / nov, 0.90, 1.25)
            
    return sku_ratio, sku_novq, cat_ratio


def pick_alpha(bucket: str, qty_series: pd.Series) -> float:
    """품목의 수요 변동성(CV) 지표에 따라 지수평활 가중치(Alpha)를 차등 부여합니다."""
    s = qty_series.dropna()
    cv = float(s.std(ddof=0) / max(s.mean(), 1e-6)) if len(s) >= 3 and s.mean() > 0 else 1.0
    
    if bucket == "high" and cv < 0.8:
        return 0.12
    if cv > 1.2:
        return 0.18
    return 0.15


def cap_spike(yhat: float, hist_prev_mean: float, bucket: str) -> float:
    """과거 평균 스케일을 상회하는 이상치(Spike)성 예측값을 보정하기 위해 임계치를 적용합니다."""
    if np.isnan(hist_prev_mean) or hist_prev_mean <= 0:
        return yhat
    mul = 1.35 if bucket == "high" else 1.60
    return min(float(yhat), float(hist_prev_mean * mul))


def dec_weight_for(pid: str, category: str, sku_novq: dict, group_value: str = None) -> float:
    """12월 연말 가중치 로직을 계산하며, 그룹 계수 및 안정성 클립을 융합합니다."""
    q = sku_novq.get(pid, None)
    if q is None:
        base = 0.25
    elif q <= 3:
        base = 0.15
    elif q >= 10:
        base = 0.30
    else:
        base = 0.25
        
    gmul = DEC_GROUP_WEIGHT.get(group_value if group_value else "other", 1.00)
    return clamp(base * gmul, 0.10, 0.35)


def compute_first_order_anchor(df: pd.DataFrame) -> tuple:
    """첫 발주 품목 및 신규 고객 유입 패턴에 기반한 시즌 앵커 스케일을 연산합니다."""
    dff = df.dropna(subset=["member_id"]).copy()
    if dff.empty:
        return 0.0, 0.0, {m: 1.0 for m in range(1, 13)}
        
    first_dt = dff.groupby("member_id")["dt"].min().rename("first_dt")
    mo = first_dt.dt.month
    mo_counts = mo.value_counts().to_dict()
    avg_ct = np.mean(list(mo_counts.values())) if mo_counts else 1.0
    
    mo_index = {}
    for m in range(1, 13):
        base = (mo_counts.get(m, 0) / avg_ct) if avg_ct > 0 else 1.0
        idx = (1.0 - SEASON_ANCHOR_GAMMA) * 1.0 + SEASON_ANCHOR_GAMMA * base
        mo_index[m] = clamp(idx, *SEASON_CLIP)
        
    first_month = first_dt.dt.to_period("M").dt.to_timestamp()
    first_cnt = first_month.value_counts().sort_index()
    
    if len(first_cnt) == 0:
        F_star = 0.0
    elif len(first_cnt) == 1:
        F_star = float(first_cnt.iloc[-1])
    else:
        F_star = float(first_cnt.tail(2).mean())
        
    df_guide = dff[dff["product_id"] == GUIDE_SKU].copy()
    if df_guide.empty:
        kappa = 1.0
    else:
        df_guide = df_guide.merge(first_dt, on="member_id", how="left")
        df_guide["is_first"] = (df_guide["dt"] == df_guide["first_dt"]).astype(int)
        if (df_guide["is_first"] == 1).any():
            kappa = float(df_guide.loc[df_guide["is_first"] == 1, "qty"].median())
            if not np.isfinite(kappa):
                kappa = 1.0
        else:
            kappa = float(df_guide["qty"].median()) if df_guide["qty"].notna().any() else 1.0
            if not np.isfinite(kappa):
                kappa = 1.0
                
    anchor_base = max(F_star * kappa, 0.0)
    return anchor_base, F_star, mo_index


def calibrate_sa_scale_low_sku(g: pd.DataFrame, category_value: str, alpha: float) -> float:
    """Low-volume 품목 대상의 시뮬레이션을 통해 예측값의 스케일 왜곡 편향을 보정(Calibration)합니다."""
    months = sorted(g["target_month"].unique())
    if len(months) < 3:
        return 1.0
    val_months = months[-2:]
    num, den = 0.0, 0.0
    
    for tm in val_months:
        hist = g[g["target_month"] < tm].copy()
        if hist.empty:
            continue
        y_ser, ex_ser = monthly_series_from_hist(hist, "qty_month", ["coupon_month", "promo_intensity"])
        yhat_raw = forecast_sarimax(y_ser, ex_ser, [tm]).get(tm, 0.0)
        lag1 = float(hist["qty_month"].iloc[-1])
        yhat = (1.0 - alpha) * float(yhat_raw) + alpha * lag1
        y_true = float(g.loc[g["target_month"] == tm, "qty_month"].iloc[0])
        num += y_true
        den += max(yhat, 0.0)
        
    if den <= 1e-9:
        return 1.0
    return clamp(num / den, 0.90, 1.10)


def predict_one_month_raw(hist_all: pd.DataFrame, tm: pd.Timestamp, bucket: str, 
                          cb_model: CatBoostRegressor, category_value: str, group_value: str) -> tuple:
    """지정 시점에 대해 단일 세그먼트 스케일 예측값을 연산합니다."""
    h_prev = hist_all[hist_all["target_month"] < tm]
    lag1 = float(h_prev["qty_month"].iloc[-1]) if len(h_prev) > 0 else 0.0

    if bucket == "high" and cb_model is not None:
        h = hist_all[hist_all["target_month"] < tm].copy()
        if h.empty:
            row = {"lag_1": 0, "lag_2": 0, "lag_3": 0, "roll_mean_3": 0, "roll_mean_6": 0,
                   "coupon_lag_1": 0, "promo_lag_1": 0, "price_lag_1": 0,
                   "year": int(tm.year), "month": int(tm.month), "is_year_end": int(tm.month == 12),
                   "cat_mean_qty": 0.0, "grp_mean_qty": 0.0,
                   "category": category_value, "group": group_value}
        else:
            row = {
                "lag_1": h["qty_month"].iloc[-1],
                "lag_2": h["qty_month"].iloc[-2] if len(h) > 1 else h["qty_month"].iloc[-1],
                "lag_3": h["qty_month"].iloc[-3] if len(h) > 2 else h["qty_month"].iloc[-1],
                "roll_mean_3": h["qty_month"].tail(3).mean(),
                "roll_mean_6": h["qty_month"].tail(min(6, len(h))).mean(),
                "coupon_lag_1": h["coupon_month"].iloc[-1] if len(h) > 0 else 0.0,
                "promo_lag_1": h["promo_intensity"].iloc[-1] if "promo_intensity" in h.columns and len(h) > 0 else 0.0,
                "price_lag_1": h["avg_price"].iloc[-1] if "avg_price" in h.columns and len(h) > 0 else 0.0,
                "year": int(tm.year), "month": int(tm.month), "is_year_end": int(tm.month == 12),
                "cat_mean_qty": float(h["cat_mean_qty"].iloc[-1]) if "cat_mean_qty" in h.columns else 0.0,
                "grp_mean_qty": float(h["grp_mean_qty"].iloc[-1]) if "grp_mean_qty" in h.columns else 0.0,
                "category": category_value,
                "group": group_value
            }
        yhat = predict_catboost(cb_model, row)
    else:
        y_series, ex_series = monthly_series_from_hist(hist_all, "qty_month", ["coupon_month", "promo_intensity"])
        yhat = forecast_sarimax(y_series, ex_series, [tm]).get(tm, 0.0)

    return float(yhat), float(lag1)


# ==========================================
# MAIN EXECUTION PIPELINE
# ==========================================
def main():
    """예측 파이프라인 전 과정을 제어하는 메인 핸들러입니다."""
    # 1. 데이터 파이프라인 개시
    raw = load_and_prepare()
    mon = make_monthly(raw)
    mon = add_features(mon)
    buckets = mark_volume(mon, thr=3.0, min_m=6)

    # 2. 템플릿 로드 및 타겟 월 설정
    tmpl = pd.read_excel(FP_TEMPLATE, dtype={"product_id": str, "target_month": str})
    if "forecast_qty" in tmpl.columns:
        tmpl = tmpl.drop(columns=["forecast_qty"])
    tmpl["product_id"] = tmpl["product_id"].map(norm_text)
    tmpl["target_month"] = tmpl["target_month"].astype(str).str.slice(0, 7)
    
    target_months = sorted({month_start(x) for x in tmpl["target_month"].unique()})
    sku_ratio, sku_novq, cat_ratio = build_uplift_maps(mon)
    anchor_base, F_star, mo_index = compute_first_order_anchor(raw)
    
    print(f"[INFO] GUIDE anchor: F*={F_star:.2f}, base={anchor_base:.2f}, "
          f"season idx(11)={mo_index.get(11, 1.0):.2f}, (12)={mo_index.get(12, 1.0):.2f}")

    # 3. 품목별 반복 순차 추론 실행
    preds = []
    for pid in tmpl["product_id"].unique():
        g = mon[mon["product_id"] == pid].dropna(subset=["qty_month"]).copy()
        if g.empty:
            for tm in target_months:
                preds.append({"product_id": pid, "target_month": tm.strftime("%Y-%m"), "forecast_qty": 0})
            continue

        category_value = g["category"].iloc[0] if "category" in g.columns else "other"
        group_value = g["group"].iloc[0] if "group" in g.columns else "other"
        bucket = buckets.loc[buckets["product_id"] == pid, "bucket"].iloc[0] if pid in set(buckets["product_id"]) else "low"

        # 모델 세그먼트 분기 처리
        cb_model = None
        if bucket == "high":
            train_cb = g.dropna(subset=["lag_1"]).copy()
            if not train_cb.empty and len(train_cb["target_month"].unique()) >= 3:
                cb_model = train_catboost(train_cb, cat_feature_names=["year", "month", "is_year_end", "category", "group"])

        alpha = pick_alpha(bucket, g["qty_month"])
        sa_scale = calibrate_sa_scale_low_sku(g, category_value, alpha) if bucket == "low" else 1.0

        hist_all = g.copy()
        last_nov_pred = None
        p90_guide = np.percentile(g["qty_month"].dropna(), 90) if (pid == GUIDE_SKU and g["qty_month"].notna().any()) else None

        # 월별 다단계(Multi-step) 순차 추론
        for tm in target_months:
            yhat_raw, lag1 = predict_one_month_raw(hist_all, tm, bucket, cb_model, category_value, group_value)
            if bucket == "low":
                yhat_raw *= sa_scale

            # 지수평활 융합
            yhat = (1.0 - alpha) * yhat_raw + alpha * lag1
            yhat = max(yhat, 0.0)

            # 12월 연말 특수성 Uplift 적용
            if tm.month == 12:
                r = sku_ratio.get(pid, cat_ratio.get(category_value, None))
                if r is not None:
                    base_nov = last_nov_pred if last_nov_pred is not None else yhat
                    w_dec = dec_weight_for(pid, category_value, sku_novq, group_value=group_value)
                    if pid == GUIDE_SKU:
                        w_dec *= 0.5
                    yhat = (1.0 - w_dec) * yhat + w_dec * (base_nov * r)
                    yhat = max(yhat, 0.0)

            # 가이드북 앵커 스케일 적용 및 상한 캡 보정
            if pid == GUIDE_SKU:
                idx = mo_index.get(int(tm.month), 1.0)
                anchor_tm = anchor_base * idx
                yhat = 0.5 * yhat + 0.5 * anchor_tm
                if p90_guide is not None and np.isfinite(p90_guide):
                    yhat = min(yhat, float(p90_guide))

            # 이상값 스파이크 방지 보정
            h_prev = hist_all[hist_all["target_month"] < tm]
            tail_mean6 = h_prev["qty_month"].tail(6).mean() if len(h_prev) > 0 else np.nan
            yhat = cap_spike(yhat, tail_mean6, bucket)

            preds.append({"product_id": pid, "target_month": tm.strftime("%Y-%m"),
                          "forecast_qty": int(round(yhat))})

            # 시계열 순차 반영을 위한 피처 가상 업데이트 (Autoregressive step)
            hist_all = pd.concat([
                hist_all,
                pd.DataFrame({
                    "product_id": [pid], "target_month": [tm], "qty_month": [yhat],
                    "coupon_month": [hist_all["coupon_month"].iloc[-1] if len(hist_all) > 0 else 0.0],
                    "promo_intensity": [hist_all["promo_intensity"].iloc[-1] if "promo_intensity" in hist_all.columns and len(hist_all) > 0 else 0.0],
                    "avg_price": [hist_all["avg_price"].iloc[-1] if "avg_price" in hist_all.columns and len(hist_all) > 0 else 0.0],
                    "cat_mean_qty": [hist_all["cat_mean_qty"].iloc[-1] if "cat_mean_qty" in hist_all.columns and len(hist_all) > 0 else 0.0],
                    "grp_mean_qty": [hist_all["grp_mean_qty"].iloc[-1] if "grp_mean_qty" in hist_all.columns and len(hist_all) > 0 else 0.0],
                    "category": [category_value], "group": [group_value]
                })
            ], ignore_index=True)

            if tm.month == 11:
                last_nov_pred = yhat

    # 4. 출력 및 정합성 저장
    pred_df = pd.DataFrame(preds)
    out = tmpl.merge(pred_df, on=["product_id", "target_month"], how="left")
    out["forecast_qty"] = out["forecast_qty"].fillna(0).astype(int)
    out = out[["product_id", "target_month", "forecast_qty"]]
    
    out.to_csv(FP_OUT, index=False, encoding="utf-8-sig")
    print(f"[OK] {FP_OUT.name} saved, rows={len(out)}")


if __name__ == "__main__":
    main()